# Gain Analysis for 03LIC_1071.PV — Deriving & Validating Process Gains

**Goal.** Estimate the process gains of `03LIC_1071.PV` (3E107 level) with respect to operator/APC handles (OP/SP) and disturbances, from historical plant data — and establish how trustworthy those numbers are.

## Why this is needed (and why the vendor gains don't cover 1071)
The customer APC is an **IPCOS DMC3** multivariable controller, split into **TR1 / TR2 / LEANGAS** sub-controllers (per the APC design doc).

- The vendor *Gain Data* file is the DMC3 **steady-state gain matrix**: rows = Manipulated Variables (MV, the `…OP`/`…SP` handles), columns = Controlled/Disturbance Variables (CV/DV, the `…PV`).
- **`03LIC_1071` is not a DMC3 CV** — it is a base-layer DCS level loop. It appears in neither the rows nor the columns of the gain matrix. So there is **no vendor gain to copy** for 1071; we must derive it from data.

## Strategy
1. **Use the APC design doc to pick physically-meaningful inputs** — the TR1 MV list and feed-forward (disturbance) list tell us which tags actually drive this train (propane compressor speed `03PIC_1013.OP`, demethaniser feed temp `03TIC_1092.SP`, total feed `02FI_1000`, …).
2. **Validate the estimator on pairs the vendor DID model** — a handful of APC MV→CV gains involve tags we also have in the parquet. Reproduce those vendor gains from data first; if the method recovers them, we trust it for 1071.
3. **Apply the validated estimator to `03LIC_1071.PV`** using its candidate inputs (own OP, supplier `03LIC_1016.OP`, `03PIC_1013.OP`, feed `02FI_1000`).

## Key caveat — closed-loop confounding
Most loops run in AUTO/cascade, and this parquet has **no MODE or SP columns**. Regressing a controlled `PV` on its *own* `OP` returns the **controller's reaction**, not the process gain (when level falls, the controller raises OP → spurious negative ‘gain’). We therefore trust **exogenous-input gains** (e.g. feed → PV) and **other-tag → CV** gains far more than self-OP gains, which we flag as closed-loop-biased.

In [1]:
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA = '/home/h604827/ControlActions/DATA'
PARQUET = f'{DATA}/PV-OP_data/03LIC_1071_JAN_2026.parquet'
GAIN_CSV = f'{DATA}/Gain Data 12Mar26(Sheet1).csv'
TARGET = '03LIC_1071.PV'

# Vendor APC steady-state gain matrix: rows = MV handles, cols = CV/DV process variables
gain_matrix = pd.read_csv(GAIN_CSV, index_col=0)
gain_matrix.index = gain_matrix.index.astype(str).str.strip()
gain_matrix.columns = gain_matrix.columns.astype(str).str.strip()

mv_has_1071 = any('1071' in r for r in gain_matrix.index)
cv_has_1071 = any('1071' in c for c in gain_matrix.columns)
print(f'Gain matrix: {gain_matrix.shape[0]} MVs (rows) x {gain_matrix.shape[1]} CVs (cols)')
print(f'03LIC_1071 present as MV? {mv_has_1071}   as CV? {cv_has_1071}')
print('=> 1071 is a base-layer DCS loop, not a DMC3 CV: no vendor gain, must derive from data.')

Gain matrix: 53 MVs (rows) x 81 CVs (cols)
03LIC_1071 present as MV? False   as CV? False
=> 1071 is a base-layer DCS loop, not a DMC3 CV: no vendor gain, must derive from data.


In [2]:
# Train-1 sub-controller variables, transcribed from the IPCOS DMC3 design doc.
TR1_MV = {
    '02PIC0003OP': 'HP comp speed',      '02HIC1147OP': 'HP recycle clamp',
    '02FIC1247OP': 'IP comp speed',      '02HIC1087OP': 'IP recycle clamp',
    '02HIC1050OP': 'LP recycle clamp',   '03PIC1013OP': 'Propane comp speed',
    '03HIC1141OP': 'Prop 1st stg clamp', '03HIC1151OP': 'Prop 2nd stg clamp',
    '03TIC1023SP': '3C101 ovhd to 3V101','03TIC1092SP': 'Demethaniser feed temp',
    '03HIC3132OP': 'IGV opening',        '03HIC3100OP': 'Recompressor',
    '03FIC3435OP': 'HYP bypass',         '03TIC1009SP': 'Demeth reboiler temp',
    '03E101FANS': '3E101 running fans',
}
TR1_FF = {
    '39TI0129PV': 'Ambient temp',   '02FI0158PV': 'HP gas to plant',
    '02FI0159PV': 'IP gas to plant','02FI0160PV': 'LP gas to plant',
    '02FI1000FF': 'Total feed Tr1', '03ZI3130PV': 'Tr1 IGV feedback',
    '03FIC3435FF': 'HYP bypass OP', '39PI0103PV': 'Lean gas disch pr',
}
print(f'TR1 manipulated variables: {len(TR1_MV)}')
print(f'TR1 feed-forward (disturbance) variables: {len(TR1_FF)}')

TR1 manipulated variables: 15
TR1 feed-forward (disturbance) variables: 8


In [3]:
# Parquet columns (schema only, no full load)
schema = pq.ParquetFile(PARQUET).schema
cols = [schema.column(i).name for i in range(len(schema))]
pv_tags = sorted({c[:-3] for c in cols if c.endswith('.PV')})
op_tags = sorted({c[:-3] for c in cols if c.endswith('.OP')})
sp_tags = sorted({c[:-3] for c in cols if c.endswith('.SP')})
print(f'Parquet: {len(cols)} cols | PV={len(pv_tags)} OP={len(op_tags)} SP={len(sp_tags)}')
print('Controllable (PV+OP):', op_tags)
print('PV-only:', sorted(set(pv_tags) - set(op_tags)))
print('NOTE: no .SP and no .MODE columns -> cannot isolate AUTO vs MANUAL from this file.')

Parquet: 46 cols | PV=28 OP=15 SP=0
Controllable (PV+OP): ['03FIC_1085', '03FIC_3415', '03LIC_1016', '03LIC_1071', '03LIC_1085', '03LIC_1094', '03LIC_1097', '03LIC_3178', '03PIC_1013', '03PIC_1068', '03PIC_1104', '03PIC_3131', '03TIC_1092', '03TIC_1142', '03TIC_1145']
PV-only: ['02FI_1000', '03FIC_3435', '03FI_1141A', '03FI_1151', '03FI_3418', '03LI_3411', '03PI_1141A', '03PI_1495', '03PI_1814', '03TI_1015', '03TI_1081', '03TI_1421', '03TI_1901']
NOTE: no .SP and no .MODE columns -> cannot isolate AUTO vs MANUAL from this file.


In [4]:
# Map an APC tag (e.g. 03PIC1013OP) to a parquet base tag (e.g. 03PIC_1013).
def apc_core(tag):
    m = re.match(r'^(.*?)(OP|SP|PV|DV|FF)$', tag)
    return m.group(1) if m else tag

def canon(s):
    return s.replace('_', '')

def relaxed(core):
    # area(2 digits) + instrument number + optional trailing letter, ignoring the letter block
    m = re.match(r'^(\d{2})[A-Z]+(\d+)([A-Z]*)$', core)
    return (m.group(1) + m.group(2) + m.group(3)) if m else None

pv_canon = {canon(t): t for t in pv_tags}
op_canon = {canon(t): t for t in op_tags}
pv_relaxed = {relaxed(canon(t)): t for t in pv_tags if relaxed(canon(t))}

def match_to_parquet(tag):
    core = apc_core(tag)
    c = canon(core)
    r = relaxed(c)
    if c in op_canon:
        return op_canon[c], 'OP-exact'
    if c in pv_canon:
        return pv_canon[c], 'PV-exact'
    if r and r in pv_relaxed:
        return pv_relaxed[r], 'PV-relaxed(verify)'
    return None, None

print('TR1 MVs available in parquet:')
for t in TR1_MV:
    hit, kind = match_to_parquet(t)
    if hit:
        print(f'   {t:13s} -> {hit:13s} [{kind}]  ({TR1_MV[t]})')
print('TR1 disturbances available in parquet:')
for t in TR1_FF:
    hit, kind = match_to_parquet(t)
    if hit:
        print(f'   {t:13s} -> {hit:13s} [{kind}]  ({TR1_FF[t]})')
print('\nCAUTION: PV-relaxed matches across instrument types can be FALSE')
print('(e.g. 03HIC1151 valve handle != 03FI_1151 flow). Trust only same-quantity matches.')

TR1 MVs available in parquet:
   03PIC1013OP   -> 03PIC_1013    [OP-exact]  (Propane comp speed)
   03HIC1151OP   -> 03FI_1151     [PV-relaxed(verify)]  (Prop 2nd stg clamp)
   03TIC1092SP   -> 03TIC_1092    [OP-exact]  (Demethaniser feed temp)
   03FIC3435OP   -> 03FIC_3435    [PV-exact]  (HYP bypass)
TR1 disturbances available in parquet:
   02FI1000FF    -> 02FI_1000     [PV-exact]  (Total feed Tr1)
   03FIC3435FF   -> 03FIC_3435    [PV-exact]  (HYP bypass OP)

CAUTION: PV-relaxed matches across instrument types can be FALSE
(e.g. 03HIC1151 valve handle != 03FI_1151 flow). Trust only same-quantity matches.


In [5]:
# Vendor gains where BOTH the MV and the CV map into our parquet -> the validation set.
long = (gain_matrix.reset_index()
        .melt(id_vars=gain_matrix.index.name or 'index', var_name='CV', value_name='gain'))
long.columns = ['MV', 'CV', 'gain']
long = long.dropna(subset=['gain'])

rows = []
for r in long.itertuples(index=False):
    mv_hit, mv_kind = match_to_parquet(r.MV)
    cv_hit, cv_kind = match_to_parquet(r.CV)
    if mv_hit and cv_hit and mv_hit != cv_hit:
        trust = 'check-names' if 'relaxed' in (str(mv_kind) + str(cv_kind)) else 'strong'
        rows.append((r.MV, mv_hit, r.CV, cv_hit, r.gain, mv_kind, cv_kind, trust))

validation = pd.DataFrame(rows, columns=['APC_MV', 'pq_MV', 'APC_CV', 'pq_CV',
                                         'APC_gain', 'MV_match', 'CV_match', 'trust'])
validation = validation.sort_values('trust').reset_index(drop=True)
print(f'Validation pairs (vendor gain available AND both tags in parquet): {len(validation)}')
validation

Validation pairs (vendor gain available AND both tags in parquet): 9


,APC_MV,pq_MV,APC_CV,pq_CV,APC_gain,MV_match,CV_match,trust
0,03PIC1013OP,03PIC_1013,03PIC1141APV,03PI_1141A,-11.235301,OP-exact,PV-relaxed(verify),check-names
1,02FI1000FF,02FI_1000,03PIC1141APV,03PI_1141A,74.377441,PV-exact,PV-relaxed(verify),check-names
2,03HIC1151OP,03FI_1151,03TIC1145PV,03TIC_1145,0.349296,PV-relaxed(verify),OP-exact,check-names
3,02FI1000FF,02FI_1000,03XISC1151DV,03FI_1151,0.035510,PV-exact,PV-relaxed(verify),check-names
4,03FIC3435OP,03FIC_3435,02FI1000PV,02FI_1000,0.006596,PV-exact,PV-exact,strong
5,02FI1000FF,02FI_1000,03TIC1145PV,03TIC_1145,0.678983,PV-exact,OP-exact,strong
6,03PIC1013OP,03PIC_1013,03TI1081PV,03TI_1081,-0.086925,OP-exact,PV-exact,strong
7,03TIC1092SP,03TIC_1092,03TI1081PV,03TI_1081,-0.444022,OP-exact,PV-exact,strong
8,02FI1000FF,02FI_1000,03TI1081PV,03TI_1081,2.039941,PV-exact,PV-exact,strong


## Empirical gain estimator

These MVs are moved (by APC or operators) almost continuously, so a discrete step test rarely applies. We instead estimate the **steady-state gain** with a **windowed multivariable Δ-regression**.

For a horizon `H` of a few process time-constants (here lag ~0–2 min, τ ~25 min, so `H = 60 min`):

`Δ_H CV(t) = Σ_i g_i · Δ_H MV_i(t) + c + ε`, where `Δ_H x(t) = x(t+H) − x(t)`

- Including **all available inputs jointly** deconfounds simultaneous moves (operators move several handles at once).
- Data is put on a strict 1-min grid (`asfreq`), so gaps/trips become NaN and are dropped — a window never spans a gap.
- Each `g_i` is the steady-state gain of `CV` w.r.t. `MV_i`, directly comparable to the vendor matrix.

**Trust:** exogenous inputs (feed `02FI_1000`) and other-tag → CV gains. **Distrust:** a controlled PV vs its **own** OP (closed-loop inversion).

In [6]:
# Load only the signals we need, on a strict 1-min grid.
needed = ['TimeStamp',
          '03PIC_1013.OP', '03TIC_1092.OP', '02FI_1000.PV',
          '03TI_1081.PV', '03TIC_1145.PV', '03PI_1141A.PV',
          '03LIC_1071.PV', '03LIC_1071.OP', '03LIC_1016.OP']
ts = pd.read_parquet(PARQUET, columns=needed)
ts['TimeStamp'] = pd.to_datetime(ts['TimeStamp'])
ts = ts.set_index('TimeStamp').sort_index()
ts = ts[~ts.index.duplicated(keep='first')].asfreq('1min')
print(f'Loaded {ts.shape[1]} signals on 1-min grid: {len(ts):,} rows, '
      f'{ts.index.min().date()} -> {ts.index.max().date()}')

Loaded 9 signals on 1-min grid: 1,824,360 rows, 2022-01-03 -> 2025-06-23


In [7]:
def estimate_gains(ts, target, inputs, H=60, thin=10):
    """Steady-state gains via H-minute multivariable difference regression.
    Returns (gains dict, n_windows, r2)."""
    cols = [target] + inputs
    diffs = pd.DataFrame({c: ts[c].shift(-H) - ts[c] for c in cols})
    diffs = diffs.iloc[::thin].dropna()
    y = diffs[target].values
    X = np.column_stack([diffs[c].values for c in inputs] + [np.ones(len(diffs))])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ coef
    r2 = 1.0 - resid.var() / y.var() if y.var() > 0 else np.nan
    return dict(zip(inputs, coef[:-1])), len(diffs), r2

In [ ]:
# Reproduce vendor gains for the validation CVs we can observe.
def base(col):
    return col.split('.')[0]

apc_lookup = {(base(r.pq_MV), base(r.pq_CV)): r.APC_gain
              for r in validation.itertuples(index=False)}

shared_inputs = ['03PIC_1013.OP', '03TIC_1092.OP', '02FI_1000.PV']
records = []
for cv in ['03TI_1081.PV', '03TIC_1145.PV', '03PI_1141A.PV']:
    gains, n, r2 = estimate_gains(ts, cv, shared_inputs, H=60, thin=10)
    for mv in shared_inputs:
        apc = apc_lookup.get((base(mv), base(cv)))
        sign_match = bool(np.sign(apc) == np.sign(gains[mv])) if apc is not None else None
        records.append({'CV': cv, 'MV': mv,
                        'derived_gain': round(float(gains[mv]), 4),
                        'APC_gain': None if apc is None else round(float(apc), 4),
                        'sign_match': sign_match, 'n_windows': n, 'r2': round(float(r2), 3)})
        
validation_results = pd.DataFrame(records)
validation_results

,CV,MV,derived_gain,APC_gain,sign_match,n_windows,r2
0,03TI_1081.PV,03PIC_1013.OP,-0.1215,-0.0869,True,173343,0.240
1,03TI_1081.PV,03TIC_1092.OP,0.0082,-0.4440,False,173343,0.240
2,03TI_1081.PV,02FI_1000.PV,1.0672,2.0399,True,173343,0.240
3,03TIC_1145.PV,03PIC_1013.OP,-0.0666,NaN,None,173343,0.023
4,03TIC_1145.PV,03TIC_1092.OP,-0.0004,NaN,None,173343,0.023
5,03TIC_1145.PV,02FI_1000.PV,-0.8896,0.6790,False,173343,0.023
6,03PI_1141A.PV,03PIC_1013.OP,-8.1548,-11.2353,True,173333,0.328
7,03PI_1141A.PV,03TIC_1092.OP,0.3917,NaN,None,173333,0.328
8,03PI_1141A.PV,02FI_1000.PV,49.2900,74.3774,True,173333,0.328


In [9]:
# Apply to the target. Inputs: own OP (closed-loop caveat), supplier 1016 OP,
# propane comp speed, and exogenous feed.
target_inputs = ['03LIC_1071.OP', '03LIC_1016.OP', '03PIC_1013.OP', '02FI_1000.PV']
g1071, n, r2 = estimate_gains(ts, TARGET, target_inputs, H=60, thin=10)
print(f'Target {TARGET}: n={n:,} windows, R^2={r2:.3f}\n')
for mv in target_inputs:
    note = ''
    if mv == '03LIC_1071.OP':
        note = '  <-- closed-loop self-OP: NOT a process gain'
    elif mv == '02FI_1000.PV':
        note = '  <-- exogenous disturbance: most trustworthy'
    print(f'   {mv:16s} g = {g1071[mv]:+8.4f}{note}')

Target 03LIC_1071.PV: n=173,349 windows, R^2=0.444

   03LIC_1071.OP    g =  -1.5577  <-- closed-loop self-OP: NOT a process gain
   03LIC_1016.OP    g =  +0.0590
   03PIC_1013.OP    g =  +0.1228
   02FI_1000.PV     g =  +2.2256  <-- exogenous disturbance: most trustworthy


## Interim findings — method validated, and why the self-gain needs a different lens

**The estimator works.** It recovers the vendor APC gains in **sign and ~50–75 % of magnitude** on the strong anchors (e.g. `03PIC_1013.OP → 03TI_1081.PV` −0.12 vs −0.087; `02FI_1000 → 03PI_1141A.PV` +49 vs +74). The shortfall is expected — omitted MVs, closed-loop attenuation, feed-forward unit/filtering — so we trust its **direction**, and magnitude to within ~2×.

**Target `03LIC_1071.PV`:**
- `02FI_1000 → 1071.PV ≈ +2.2` — exogenous feed disturbance, sensible and trustworthy (more feed → higher level).
- `03LIC_1071.OP → 1071.PV ≈ −1.6` — **not a process gain**: it's reverse-causation (in closed loop the controller raises OP *because* the level is falling).

**So the self-gain needs a different lens.** Two facts shape the rest of the notebook:
1. **`03LIC_1071` is an integrating level** — the valve sets the *rate* of level change, not a steady level, so a steady-state `∂PV/∂OP` is the wrong quantity to chase.
2. **MODE and SP aren't in the parquet, but they are in the events log** (`Description ∈ {OP, SP, MODE}`) — so we can isolate MANUAL operation and attribute operator moves directly, with **no data re-export needed**.

The next section uses the events log to measure the **response time and effect on `03LIC_1071.PV`** of the handles operators actually move around the alarm.


## Cross-tag response time & effect on `03LIC_1071.PV`

Operators move several propane-loop handles around 1071 PVLO events. Here we quantify, for each, **(i) how long 1071.PV takes to respond** and **(ii) the direction/size of the effect** on 1071.PV.

**Tags:** `03LIC_1016`, `03LIC_1034`, `03HIC_1141`, `03HIC_1151`, `03PIC_1013`.

**Lever & data source per tag.** The parquet has no `.SP`/`.MODE`, so handles absent from it are rebuilt from the events log by forward-filling the recorded `Value`:
- `03LIC_1016.OP`, `03PIC_1013.OP` — from the parquet (continuous).
- `03LIC_1034.SP` (operators ride its setpoint), `03HIC_1141.OP`, `03HIC_1151.OP` — reconstructed from events.

**Two views, because attribution is genuinely hard here** — 1071 is an *integrating* level that is already falling during an alarm, while operators move many handles at once:
1. **Detrended event response** — for *isolated* moves (no other move of the same tag, **and no 1071 self OP/SP move within ±15 min**), fit the pre-action level trend and measure when 1071.PV departs >3σ from that extrapolated trend. This yields a response **lag** and an effect **direction** with the dominant reverse-causation (operators acting on 1071 itself) removed.
2. **Deconfounded joint Δ-regression near alarms** — regress Δ₃₀(1071.PV) on the simultaneous Δ₃₀ of *all* handles + feed + `03LIC_1071.OP`, within ±2 h of a PVLO start, so co-moving handles don't steal each other's credit.


In [10]:
# ── Cross-tag setup: rebuild the handles operators moved around 1071 alarms ────
EVENTS = f'{DATA}/trip_filtered_events_dedup.csv'
ev = pd.read_csv(EVENTS, low_memory=False,
                 usecols=['VT_Start', 'Source', 'ConditionName', 'Description',
                          'PrevValue', 'Value', 'Action'])
ev['VT_Start'] = pd.to_datetime(ev['VT_Start'])
chg = ev[ev['ConditionName'] == 'CHANGE'].copy()

pv = ts[TARGET]   # 03LIC_1071.PV on the 1-min grid

def reconstruct_handle(tag, desc):
    """Rebuild a handle's trajectory on the 1-min grid from its recorded CHANGE values."""
    s = chg[(chg['Source'] == tag) & (chg['Description'] == desc)].copy()
    s['Value'] = pd.to_numeric(s['Value'], errors='coerce')
    s = s.dropna(subset=['Value'])
    s['t'] = s['VT_Start'].dt.floor('min')
    s = s.sort_values('t').drop_duplicates('t', keep='last')
    return pd.Series(s['Value'].values, index=s['t']).reindex(ts.index, method='ffill')

# Lever operators actually move for each tag, and its trajectory
# (parquet where available, else reconstructed from events).
CROSS_LEVERS = {
    '03LIC_1016': ('OP', ts['03LIC_1016.OP']),
    '03PIC_1013': ('OP', ts['03PIC_1013.OP']),
    '03LIC_1034': ('SP', reconstruct_handle('03LIC_1034', 'SP')),
    '03HIC_1141': ('OP', reconstruct_handle('03HIC_1141', 'OP')),
    '03HIC_1151': ('OP', reconstruct_handle('03HIC_1151', 'OP')),
}

# Minutes when 1071's OWN OP/SP was moved — used to strip reverse-causation.
self_moves = np.array(sorted(set(
    chg[(chg['Source'] == '03LIC_1071') & (chg['Description'].isin(['OP', 'SP']))]['VT_Start'].dt.floor('min')
))).astype('datetime64[m]')

# Near-alarm mask: within ±2 h of a 1071 PVLO start.
pvlo = ev[(ev['Source'] == '03LIC_1071') & (ev['ConditionName'] == 'PVLO') & (ev['Action'].isna())]
alarm_starts = np.sort(pvlo['VT_Start'].dt.floor('min').values.astype('datetime64[m]'))
grid_m = ts.index.values.astype('datetime64[m]')
_pos = np.searchsorted(alarm_starts, grid_m)
near_alarm = np.zeros(len(ts), bool)
for _k in (-1, 0):
    _j = np.clip(_pos + _k, 0, len(alarm_starts) - 1)
    near_alarm |= np.abs((grid_m - alarm_starts[_j]) / np.timedelta64(1, 'm')) <= 120

print(f"events CHANGE rows: {len(chg):,} | 1071 self-move minutes: {len(self_moves):,}")
print(f"1071 PVLO starts: {len(alarm_starts):,} | grid minutes near an alarm: {near_alarm.sum():,}")
for tag, (lever, series) in CROSS_LEVERS.items():
    src = 'parquet' if tag in ('03LIC_1016', '03PIC_1013') else 'events-reconstructed'
    print(f"  {tag} [{lever:2s}] <- {src:20s}: {series.notna().sum():,} non-null grid points")


events CHANGE rows: 115,857 | 1071 self-move minutes: 779
1071 PVLO starts: 1,406 | grid minutes near an alarm: 112,287
  03LIC_1016 [OP] <- parquet             : 1,737,582 non-null grid points
  03PIC_1013 [OP] <- parquet             : 1,737,574 non-null grid points
  03LIC_1034 [SP] <- events-reconstructed: 1,824,360 non-null grid points
  03HIC_1141 [OP] <- events-reconstructed: 1,824,360 non-null grid points
  03HIC_1151 [OP] <- events-reconstructed: 1,824,360 non-null grid points


In [11]:
# ── View 1: detrended event response time + direction ─────────────────────────
def _self_move_near(t_m, win=15):
    i = np.searchsorted(self_moves, t_m)
    for j in (i - 1, i):
        if 0 <= j < len(self_moves) and abs((t_m - self_moves[j]) / np.timedelta64(1, 'm')) <= win:
            return True
    return False

def _mins(idx, t):
    return np.asarray((idx - t).total_seconds()) / 60.0

def detrended_response(tag, desc, H=45, pre=15, iso=20, min_step=0.5):
    """Lag until 1071.PV departs >3σ from its pre-action trend, for isolated moves
    with no 1071 self OP/SP move within ±15 min."""
    s = chg[(chg['Source'] == tag) & (chg['Description'] == desc)].copy()
    s['P'] = pd.to_numeric(s['PrevValue'], errors='coerce')
    s['V'] = pd.to_numeric(s['Value'], errors='coerce')
    s = s.dropna(subset=['P', 'V'])
    s['dX'] = s['V'] - s['P']
    s = s[s['dX'].abs() >= min_step]
    s['t'] = s['VT_Start'].dt.floor('min')
    s = s.sort_values('t')
    tt = s['t'].values.astype('datetime64[m]')
    iso_ok = np.array([(np.abs((tt - ti) / np.timedelta64(1, 'm')) < iso).sum() == 1 for ti in tt])
    s = s[iso_ok]
    s = s[~np.array([_self_move_near(np.datetime64(t, 'm')) for t in s['t']], dtype=bool)]
    lags, effs, dXs = [], [], []
    for _, r in s.iterrows():
        t = r['t']
        prew = pv.loc[t - pd.Timedelta(minutes=pre):t].dropna()
        postw = pv.loc[t + pd.Timedelta(minutes=1):t + pd.Timedelta(minutes=H)].dropna()
        if len(prew) < 8 or len(postw) < 10:
            continue
        xm = _mins(prew.index, t)
        a, b = np.polyfit(xm, prew.values, 1)              # pre-action level trend
        rstd = max((prew.values - (a * xm + b)).std(), 0.05)
        xp = _mins(postw.index, t)
        resid = postw.values - (a * xp + b)                # departure from extrapolated trend
        over = np.where(np.abs(resid) > 3 * rstd)[0]
        lags.append(xp[over[0]] if len(over) else np.nan)
        effs.append(resid[-1])
        dXs.append(r['dX'])
    effs, dXs, lags = np.array(effs), np.array(dXs), np.array(lags)
    n = len(effs)
    if n < 5:
        return dict(tag=tag, lever=desc, n=n, med_lag_min=np.nan,
                    pct_detected=np.nan, pct_same_dir=np.nan)
    return dict(tag=tag, lever=desc, n=n,
                med_lag_min=round(float(np.nanmedian(lags)), 1),
                pct_detected=round(100 * float(np.mean(~np.isnan(lags)))),
                pct_same_dir=round(100 * float(np.mean(np.sign(dXs) == np.sign(np.nan_to_num(effs))))))

response_tbl = pd.DataFrame([detrended_response(tag, lever) for tag, (lever, _) in CROSS_LEVERS.items()])
print("View 1 — detrended event response (isolated; reverse-causation removed):")
print("  med_lag_min  = median minutes until 1071.PV departs >3σ from its pre-action trend")
print("  pct_same_dir = % of moves where 1071.PV departs in the SAME direction as the handle")
response_tbl


View 1 — detrended event response (isolated; reverse-causation removed):
  med_lag_min  = median minutes until 1071.PV departs >3σ from its pre-action trend
  pct_same_dir = % of moves where 1071.PV departs in the SAME direction as the handle


,tag,lever,n,med_lag_min,pct_detected,pct_same_dir
0,03LIC_1016,OP,1,NaN,NaN,NaN
1,03PIC_1013,OP,298,6.0,91.0,41.0
2,03LIC_1034,SP,942,5.0,92.0,52.0
3,03HIC_1141,OP,92,4.0,97.0,29.0
4,03HIC_1151,OP,1448,5.0,94.0,47.0


In [12]:
# ── View 2: deconfounded joint Δ-regression near alarms ───────────────────────
def joint_cross_gains(H=30, near_only=True):
    """Δ_H(1071.PV) regressed on simultaneous Δ_H of all handles + feed + self-OP.
    Restricting to ±2 h of a PVLO start keeps it in the alarm regime; fitting all
    inputs jointly stops co-moving handles from stealing each other's credit."""
    inputs = {tag: series for tag, (lever, series) in CROSS_LEVERS.items()}
    inputs['02FI_1000 (feed)'] = ts['02FI_1000.PV']
    inputs['03LIC_1071.OP (self)'] = ts['03LIC_1071.OP']
    D = pd.DataFrame({'y': pv.shift(-H) - pv})
    for nm, s in inputs.items():
        D[nm] = s.shift(-H) - s
    mask = near_alarm if near_only else np.ones(len(ts), bool)
    D = D[mask].dropna()
    X = np.column_stack([D[nm].values for nm in inputs] + [np.ones(len(D))])
    coef, *_ = np.linalg.lstsq(X, D['y'].values, rcond=None)
    resid = D['y'].values - X @ coef
    r2 = 1 - resid.var() / D['y'].values.var()
    out = pd.DataFrame({'input': list(inputs.keys()),
                        'effect_on_1071PV_per_unit_move': np.round(coef[:-1], 4)})
    return out, len(D), round(float(r2), 3)

cross_gains, n_win, r2_cross = joint_cross_gains(H=30, near_only=True)
print(f"View 2 — joint Δ30 regression within ±2h of a 1071 PVLO start: n={n_win:,} windows, R²={r2_cross}")
print("(positive => raising the handle pushes 1071.PV up; self-OP term carries the reverse-causation)")
cross_gains


View 2 — joint Δ30 regression within ±2h of a 1071 PVLO start: n=109,800 windows, R²=0.496
(positive => raising the handle pushes 1071.PV up; self-OP term carries the reverse-causation)


,input,effect_on_1071PV_per_unit_move
0,03LIC_1016,0.0753
1,03PIC_1013,0.0071
2,03LIC_1034,0.1656
3,03HIC_1141,-0.4634
4,03HIC_1151,0.0111
5,02FI_1000 (feed),0.8194
6,03LIC_1071.OP (self),-1.7079


## Cross-tag findings

**Response time is fast and similar across handles** — once a handle moves, `03LIC_1071.PV` departs from its prior trend within a **median ~4–6 min** (≥91 % of isolated moves produce a detectable >3σ departure within 45 min).

| Tag | Lever | Role (APC design doc) | Resp. lag | Joint cross-gain /unit | Net effect on 1071 |
|---|---|---|---|---|---|
| `03HIC_1141` | OP | Prop 1st-stg recycle clamp | ~4 min | **−0.46** (29 % same-dir) | **Lowers** 1071 (clearest) |
| `03LIC_1034` | SP | Level (demeth area) | ~5 min | +0.17 (52 %) | Mild raise |
| `03LIC_1016` | OP | Supplier level to 1071 | –¹ | +0.08¹ | Raise (supplies 1071) |
| `03HIC_1151` | OP | Prop 2nd-stg recycle clamp | ~5 min | +0.01 (47 %) | ≈ none |
| `03PIC_1013` | OP | Propane comp speed | ~6 min | +0.01 (41 %) | ≈ none once deconfounded |

For reference in the same fit: **feed `02FI_1000` +0.82** (dominant driver, raises level) and **self `03LIC_1071.OP` −1.71** (reverse-causation term, *not* a gain).

**Reading it**
- **`03HIC_1141` (1st-stage recycle clamp) is the clearest cross-lever**: opening it *lowers* 1071 (−0.46; 71 % of isolated moves push the level opposite to the handle). Physically — more recycle → less propane compression/cooling → less condensation into 3E107 → level falls.
- **`03LIC_1016` raises 1071** (it supplies the vessel), but operators almost always move it *together* with 1071 itself, so only one move is cleanly isolable — the +0.08 is from the joint regression, not the event view.
- **`03PIC_1013` and `03HIC_1151` show little independent effect** on 1071 once co-moving handles and feed are accounted for; most of their raw apparent effect was reverse-causation / co-movement.

¹ `03LIC_1016` is collinear with `03LIC_1071`'s own actions → event isolation yields n=1; trust only the joint-regression value.

**Caveats.** 1071 is integrating, so these are ~30-min effects, not steady-state gains; the self-OP term still carries reverse-causation that can bias magnitudes; cross-gains are associations in the alarm regime and are most reliable as **direction** indicators.
